## Xây dựng máy tính chơi caro 5x5 với chiến lược Minimax và cắt tỉa Alpha-Beta
Mục tiêu:

Hiểu và cài đặt thuật toán Minimax trong tìm kiếm có đối thủ, đồng thời tối ưu hóa hiệu suất bằng cách sử dụng cắt tỉa Alpha-Beta.

Biết cách biểu diễn bàn cờ có kích thước lớn hơn (5x5), và các hàm kiểm tra trạng thái thắng/thua/hòa.

# Thực hành viết chương trình Python cho phép người chơi tương tác với máy tính.

 Mô tả:

Trò chơi caro 5x5 (tic-tac-toe) với các quy tắc tương tự như trò chơi 3x3, nhưng có bàn cờ lớn hơn và điều kiện thắng được mở rộng thành việc có 5 ký hiệu liên tiếp (X hoặc O) trên một hàng, cột, hoặc đường chéo.

Người chơi: ký hiệu X.

Máy tính: ký hiệu O.

Máy tính chọn nước đi dựa trên chiến lược Minimax, đồng thời sử dụng cắt tỉa Alpha-Beta để tối ưu hóa việc tìm kiếm và giảm số lượng nhánh cần phải duyệt.

 Yêu cầu:

Biểu diễn bàn cờ:

Bàn cờ có kích thước 5x5.

X = 1, O = -1, Trống = 0.

Viết các hàm cơ bản:

print_board(board): In bàn cờ.

check_winner(board): Kiểm tra kết quả ván cờ:

-Trả về 1 nếu X thắng.

-Trả về -1 nếu O thắng.

-Trả về 0 nếu hòa.

-Trả về None nếu chưa kết thúc.

is_moves_left(board): Kiểm tra còn ô trống không.

Cài đặt Minimax với Alpha-Beta:

Người chơi X: maximize điểm (tối đa hóa kết quả).

Máy tính O: minimize điểm (tối thiểu hóa kết quả).

Điểm số:

+1: nếu X thắng.

-1: nếu O thắng.

0: nếu hòa.

Tương tác với người chơi:

Người chơi nhập nước đi (hàng, cột). Lưu ý, kiểm tra nước đi hợp lệ (trong phạm vi bàn cờ và ô chưa bị chiếm).

Máy tính dùng Minimax với cắt tỉa Alpha-Beta để chọn nước đi tối ưu.

In ra bàn cờ sau mỗi lượt của người chơi và máy tính.

In [2]:
import math

# ================== Cấu hình ==================
N = 5               # Kích thước bàn cờ (5x5)
WIN_CONDITION = 4   # Số quân liên tiếp để thắng
MAX_DEPTH = 3       # Giới hạn độ sâu tìm kiếm

# Biểu diễn
X = 1       # Người
O = -1      # Máy
EMPTY = 0

In [3]:
# ================== Hàm cơ bản ==================
def print_board(board):
    """In bàn cờ ra màn hình."""
    symbols = {X: "X", O: "O", EMPTY: "."}
    for row in board:
        print(" ".join(symbols[cell] for cell in row))
    print()


def is_moves_left(board):
    """Kiểm tra còn ô trống không."""
    for row in board:
        if EMPTY in row:
            return True
    return False


def check_winner(board):
    """Kiểm tra thắng/thua/hòa."""
    N = len(board)
    for i in range(N):
        for j in range(N):
            if board[i][j] == EMPTY:
                continue
            player = board[i][j]

            # ngang
            if j + WIN_CONDITION <= N and all(board[i][j+k] == player for k in range(WIN_CONDITION)):
                return player

            # dọc
            if i + WIN_CONDITION <= N and all(board[i+k][j] == player for k in range(WIN_CONDITION)):
                return player

            # chéo xuống phải
            if i + WIN_CONDITION <= N and j + WIN_CONDITION <= N and all(board[i+k][j+k] == player for k in range(WIN_CONDITION)):
                return player

            # chéo xuống trái
            if i + WIN_CONDITION <= N and j - WIN_CONDITION + 1 >= 0 and all(board[i+k][j-k] == player for k in range(WIN_CONDITION)):
                return player

    if not is_moves_left(board):
        return 0  # hòa
    return None  # chưa kết thúc

In [4]:
# ================== Heuristic ==================
def score_line(line):
    """Đánh giá một dãy liên tiếp (ngang/dọc/chéo)."""
    score = 0
    count_X = line.count(X)
    count_O = line.count(O)

    if count_X > 0 and count_O == 0:  # chỉ có X
        if count_X == WIN_CONDITION - 1:
            score += 1000
        elif count_X == WIN_CONDITION - 2:
            score += 100
        else:
            score += count_X
    elif count_O > 0 and count_X == 0:  # chỉ có O
        if count_O == WIN_CONDITION - 1:
            score -= 1000
        elif count_O == WIN_CONDITION - 2:
            score -= 100
        else:
            score -= count_O

    return score


def evaluate(board):
    """Hàm heuristic đánh giá trạng thái trung gian."""
    score = 0
    N = len(board)

    # Kiểm tra hàng
    for i in range(N):
        for j in range(N - WIN_CONDITION + 1):
            line = board[i][j:j + WIN_CONDITION]
            score += score_line(line)

    # Kiểm tra cột
    for j in range(N):
        for i in range(N - WIN_CONDITION + 1):
            line = [board[i+k][j] for k in range(WIN_CONDITION)]
            score += score_line(line)

    # Kiểm tra chéo xuống phải
    for i in range(N - WIN_CONDITION + 1):
        for j in range(N - WIN_CONDITION + 1):
            line = [board[i+k][j+k] for k in range(WIN_CONDITION)]
            score += score_line(line)

    # Kiểm tra chéo xuống trái
    for i in range(N - WIN_CONDITION + 1):
        for j in range(WIN_CONDITION - 1, N):
            line = [board[i+k][j-k] for k in range(WIN_CONDITION)]
            score += score_line(line)

    return score

In [5]:
# ================== Minimax ==================
def minimax(board, depth, alpha, beta, maximizingPlayer):
    """Thuật toán Minimax có depth limit + Alpha-Beta + heuristic."""
    winner = check_winner(board)
    if winner is not None:
        return winner * 10000  # điểm cực lớn khi có kết quả

    if depth == MAX_DEPTH:
        return evaluate(board)

    if maximizingPlayer:  # Người (X)
        maxEval = -math.inf
        for i in range(N):
            for j in range(N):
                if board[i][j] == EMPTY:
                    board[i][j] = X
                    eval = minimax(board, depth + 1, alpha, beta, False)
                    board[i][j] = EMPTY
                    maxEval = max(maxEval, eval)
                    alpha = max(alpha, eval)
                    if beta <= alpha:
                        return maxEval
        return maxEval
    else:  # Máy (O)
        minEval = math.inf
        for i in range(N):
            for j in range(N):
                if board[i][j] == EMPTY:
                    board[i][j] = O
                    eval = minimax(board, depth + 1, alpha, beta, True)
                    board[i][j] = EMPTY
                    minEval = min(minEval, eval)
                    beta = min(beta, eval)
                    if beta <= alpha:
                        return minEval
        return minEval


def find_best_move(board, player):
    """Tìm nước đi tối ưu cho player."""
    bestMove = (-1, -1)
    if player == X:  # maximize
        bestVal = -math.inf
        for i in range(N):
            for j in range(N):
                if board[i][j] == EMPTY:
                    board[i][j] = X
                    moveVal = minimax(board, 0, -math.inf, math.inf, False)
                    board[i][j] = EMPTY
                    if moveVal > bestVal:
                        bestVal = moveVal
                        bestMove = (i, j)
    else:  # minimize
        bestVal = math.inf
        for i in range(N):
            for j in range(N):
                if board[i][j] == EMPTY:
                    board[i][j] = O
                    moveVal = minimax(board, 0, -math.inf, math.inf, True)
                    board[i][j] = EMPTY
                    if moveVal < bestVal:
                        bestVal = moveVal
                        bestMove = (i, j)
    return bestMove


In [6]:
# ================== Game Loop ==================
def play_game():
    """Chơi Caro: Người vs Máy."""
    board = [[EMPTY] * N for _ in range(N)]

    print(f"=== Caro {N}x{N}, thắng {WIN_CONDITION} quân liên tiếp ===")
    print_board(board)

    while True:
        # Người chơi đi
        while True:
            try:
                move = input("Nhập nước đi (hàng cột, ví dụ 0 0): ")
                i, j = map(int, move.split())
                if 0 <= i < N and 0 <= j < N and board[i][j] == EMPTY:
                    board[i][j] = X
                    break
                else:
                    print("Nước đi không hợp lệ. Nhập lại!")
            except:
                print("Lỗi nhập, nhập lại!")

        print_board(board)
        result = check_winner(board)
        if result is not None:
            break

        # Máy đi
        print("Máy đang suy nghĩ...")
        i, j = find_best_move(board, O)
        board[i][j] = O
        print_board(board)
        result = check_winner(board)
        if result is not None:
            break

    # Kết thúc
    if result == X:
        print("Bạn thắng! 🎉")
    elif result == O:
        print("Máy thắng! 🤖")
    else:
        print("Hòa!")


if __name__ == "__main__":
    play_game()

=== Caro 5x5, thắng 4 quân liên tiếp ===
. . . . .
. . . . .
. . . . .
. . . . .
. . . . .

Nhập nước đi (hàng cột, ví dụ 0 0): 2 2
. . . . .
. . . . .
. . X . .
. . . . .
. . . . .

Máy đang suy nghĩ...
. . . . .
. O . . .
. . X . .
. . . . .
. . . . .

Nhập nước đi (hàng cột, ví dụ 0 0): 2 1
. . . . .
. O . . .
. X X . .
. . . . .
. . . . .

Máy đang suy nghĩ...
. . . . .
. O . . .
. X X O .
. . . . .
. . . . .

Nhập nước đi (hàng cột, ví dụ 0 0): 1 2
. . . . .
. O X . .
. X X O .
. . . . .
. . . . .

Máy đang suy nghĩ...
. . . . .
. O X . .
. X X O .
. . O . .
. . . . .

Nhập nước đi (hàng cột, ví dụ 0 0): 1 2
Nước đi không hợp lệ. Nhập lại!
Nhập nước đi (hàng cột, ví dụ 0 0): 1 3
. . . . .
. O X X .
. X X O .
. . O . .
. . . . .

Máy đang suy nghĩ...
. . . . .
. O X X .
. X X O .
. O O . .
. . . . .

Nhập nước đi (hàng cột, ví dụ 0 0): 3 3
. . . . .
. O X X .
. X X O .
. O O X .
. . . . .

Máy đang suy nghĩ...
. . . O .
. O X X .
. X X O .
. O O X .
. . . . .

Nhập nước đi (hàng cộ